# MegatronBridge API — Colab Quickstart

This notebook runs the full MegatronBridge API stack on a Colab H100 and walks through:
1. Environment verification
2. Installing dependencies
3. Starting the API server
4. Exposing it publicly via ngrok
5. Submitting a checkpoint import job
6. Streaming real-time logs via WebSocket
7. Monitoring GPU telemetry via the progress stream

**Required runtime: H100 (or A100)**  
Runtime → Change runtime type → Hardware accelerator: H100

[![GitHub](https://img.shields.io/badge/GitHub-Nvidia--megatron--bridge--API-black)](https://github.com/Doondi-Ashlesh/Nvidia-megatron-bridge-API)

## Step 1 — Verify environment

Requirements: Python ≥ 3.12, PyTorch ≥ 2.7, CUDA ≥ 12.8

In [ ]:
import sys
import subprocess

import torch

print(f"Python:  {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print()

# Check requirements
py_ok = sys.version_info >= (3, 12)
pt_ok = tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) >= (2, 7)
cuda_ok = torch.version.cuda is not None and float(torch.version.cuda) >= 12.8

print(f"Python ≥ 3.12:    {'✅' if py_ok else '❌'}")
print(f"PyTorch ≥ 2.7:    {'✅' if pt_ok else '❌'}")
print(f"CUDA ≥ 12.8:      {'✅' if cuda_ok else '❌'}")

if not all([py_ok, pt_ok, cuda_ok]):
    raise RuntimeError("Requirements not met. Switch to H100 runtime: Runtime → Change runtime type")

# GPU info
result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print()
print(result.stdout)

## Step 2 — Clone repository and install dependencies

In [ ]:
import os

# Clone
if not os.path.exists("/content/Nvidia-megatron-bridge-API"):
    !git clone https://github.com/Doondi-Ashlesh/Nvidia-megatron-bridge-API /content/Nvidia-megatron-bridge-API

os.chdir("/content/Nvidia-megatron-bridge-API")
print("Working directory:", os.getcwd())

In [ ]:
# Install API server dependencies (fast — no GPU deps)
!pip install -e ".[dev]" -q
print("✅ API server dependencies installed")

In [ ]:
# Install megatron-bridge
# Requires --no-build-isolation because TransformerEngine must see the already-installed
# CUDA toolkit and PyTorch headers during its native extension build.
# This takes 8-12 minutes on first run.
print("Installing megatron-bridge (this takes ~10 minutes)...")
!pip install setuptools pybind11 wheel_stub -q
!pip install --no-build-isolation megatron-bridge -q
print("✅ megatron-bridge installed")

## Step 3 — Configure environment

In [ ]:
import os

os.environ["CHECKPOINTS_ROOT"] = "/content/checkpoints"
os.environ["LOGS_ROOT"]        = "/content/logs"
os.environ["DATABASE_URL"]     = "sqlite+aiosqlite:////content/megatronbridge.db"
os.environ["LOG_LEVEL"]        = "INFO"

# Optional: set an API key to protect the endpoints
# os.environ["API_KEY"] = "my-secret-key"

!mkdir -p /content/checkpoints /content/logs
print("✅ Environment configured")
print(f"  Checkpoints: {os.environ['CHECKPOINTS_ROOT']}")
print(f"  Logs:        {os.environ['LOGS_ROOT']}")
print(f"  Database:    {os.environ['DATABASE_URL']}")

## Step 4 — Start the API server

In [ ]:
import subprocess
import time
import urllib.request

# Start uvicorn as a background subprocess
server_proc = subprocess.Popen(
    ["python", "-m", "uvicorn", "app.main:app",
     "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

# Wait for startup
print("Starting API server", end="")
for _ in range(15):
    time.sleep(1)
    print(".", end="", flush=True)
    try:
        resp = urllib.request.urlopen("http://localhost:8000/health", timeout=2)
        print(f"\n✅ Server is up: {resp.read().decode()}")
        break
    except Exception:
        continue
else:
    # Print server output for debugging
    import io
    out = server_proc.stdout.read1(4096)
    print(f"\n❌ Server did not start. Output:\n{out.decode()}")

In [ ]:
import requests
import json

BASE = "http://localhost:8000"

# System info — shows GPU name, CUDA version, supported models
resp = requests.get(f"{BASE}/v1/system/info")
print(json.dumps(resp.json(), indent=2))

## Step 5 — Expose publicly via ngrok (optional)

Skip this step if you only need to call the API from within the notebook.

In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok

# If you have a free ngrok account, set your auth token to avoid connection limits:
# ngrok.set_auth_token("your_ngrok_authtoken")

tunnel = ngrok.connect(8000)
PUBLIC_URL = tunnel.public_url
print(f"✅ Public URL: {PUBLIC_URL}")
print(f"   API docs:   {PUBLIC_URL}/docs")
print(f"   Health:     {PUBLIC_URL}/health")

## Step 6 — Submit a checkpoint import job

We'll use `facebook/opt-125m` — a small, public model (no HuggingFace token required).  
For gated models (Llama 3, Mistral, etc.), add `"hf_token": "hf_..."` to the request body.

In [ ]:
import requests
import json

BASE = "http://localhost:8000"

resp = requests.post(f"{BASE}/v1/checkpoints/import", json={
    "source_path": "facebook/opt-125m",
    "target_name": "opt-125m-megatron",
    # "hf_token": "hf_..."   # uncomment for gated models
})

print("Response:", resp.status_code)
print(json.dumps(resp.json(), indent=2))

job_id = resp.json()["job_id"]
print(f"\nJob ID: {job_id}")

## Step 7 — Stream logs in real-time

In [ ]:
import websocket
import threading
import json

def stream_logs(job_id):
    """Stream log lines from the WebSocket endpoint until job completes."""
    ws = websocket.WebSocket()
    ws.connect(f"ws://localhost:8000/v1/ws/jobs/{job_id}/logs")
    print(f"--- Log stream for job {job_id} ---")
    while True:
        msg = ws.recv()
        if not msg:
            break
        try:
            data = json.loads(msg)
            if data.get("type") == "stream_end":
                print(f"\n--- Stream ended (status: {data['status']}) ---")
                break
        except json.JSONDecodeError:
            print(msg)  # plain log line from the worker
    ws.close()

# Run in a background thread so the notebook cell doesn't block
log_thread = threading.Thread(target=stream_logs, args=(job_id,), daemon=True)
log_thread.start()
print("Log stream started in background thread. Output will appear below.")

## Step 8 — Poll job status and GPU telemetry

In [ ]:
import requests
import time
import json

BASE = "http://localhost:8000"

while True:
    resp = requests.get(f"{BASE}/v1/jobs/{job_id}")
    job  = resp.json()

    status   = job["status"]
    progress = job.get("progress") or {}

    print(f"Status: {status}", end="")

    if progress:
        step  = progress.get("step", "?")
        total = progress.get("total_steps", "?")
        loss  = progress.get("loss", "?")
        gpus  = progress.get("gpus", [])
        print(f" | Step: {step}/{total} | Loss: {loss}", end="")
        if gpus:
            g = gpus[0]
            print(f" | GPU util: {g.get('util_pct')}% | Mem: {g.get('mem_used_gb'):.1f}/{g.get('mem_total_gb'):.0f} GB | Temp: {g.get('temp_c')}°C", end="")

    print()

    if status in ("completed", "failed", "cancelled"):
        if status == "failed":
            print(f"\nError: {job.get('error')}")
        else:
            print(f"\n✅ Job {status} successfully")
        break

    time.sleep(5)

## Step 9 — View registered checkpoints

In [ ]:
resp = requests.get(f"{BASE}/v1/checkpoints")
print(json.dumps(resp.json(), indent=2))

## Step 10 — Launch a LoRA fine-tuning job (optional)

Requires a Megatron-format checkpoint from the import step above, and a dataset directory.

In [ ]:
# Only run this if the import job above completed successfully
# and you have a dataset at /content/datasets/my_data

resp = requests.post(f"{BASE}/v1/peft/lora", json={
    "pretrained_checkpoint": "/content/checkpoints/opt-125m-megatron",
    "dataset_path": "/content/datasets/my_data",
    "output_dir": "/content/checkpoints/opt-125m-lora",
    "num_gpus": 1,          # Colab has 1 GPU
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_target_modules": ["linear_qkv", "linear_proj"],
})

print(resp.status_code)
print(json.dumps(resp.json(), indent=2))

## Cleanup

In [ ]:
# Stop the API server
server_proc.terminate()
server_proc.wait()
print("✅ API server stopped")

# Kill ngrok tunnel (if started)
try:
    ngrok.kill()
    print("✅ ngrok tunnel closed")
except Exception:
    pass